## Breast Cancer

In [1]:
from __future__ import annotations
import numpy as np
import torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)


### Data loading and preprocessing

In [2]:
data = fetch_openml(data_id=15, as_frame=True) 
df = data.frame
df.head()

,Clump_Thickness,Cell_Size_Uniformity,Cell_Shape_Uniformity,Marginal_Adhesion,Single_Epi_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,benign
1,5,4,4,5,7,10.0,3,2,1,benign
2,3,1,1,1,2,2.0,3,1,1,benign
3,6,8,8,1,3,4.0,3,7,1,benign
4,4,1,1,3,2,1.0,3,1,1,benign


In [3]:
target_col = "Class"

# check the unique values in the target column
print(df[target_col].unique())

# turn the target column into binary values
df[target_col] = df[target_col].apply(lambda x: 1 if x == 'malignant' else 0)

['benign', 'malignant']
Categories (2, object): ['benign', 'malignant']


In [4]:
# delete the rows with nan values
print(df.isna().sum())
df = df.dropna()

Clump_Thickness           0
Cell_Size_Uniformity      0
Cell_Shape_Uniformity     0
Marginal_Adhesion         0
Single_Epi_Cell_Size      0
Bare_Nuclei              16
Bland_Chromatin           0
Normal_Nucleoli           0
Mitoses                   0
Class                     0
dtype: int64


In [5]:
X_raw = df.drop(columns=[target_col])
y_raw = df[target_col]

X_np = X_raw.to_numpy()
y_np = y_raw.to_numpy()

scaler_x = StandardScaler()

x_scaled = torch.tensor(scaler_x.fit_transform(X_np), dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)

x_train, x_temp, y_train, y_temp = train_test_split(x_scaled, y, test_size=0.4, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

### train the BL model

In [6]:
from __future__ import annotations
import pandas as pd
from blnetwork.model import BLDeep
from blnetwork.training import OptimConfig, TrainConfig, DiscreteTrainer, ExportConfig

seed = 42
device = "cpu"

column_names = X_raw.columns.tolist()
input_names = column_names + [target_col]
feature_df = pd.DataFrame(columns=input_names)

model = BLDeep(hidden_dims=[2,1], first_act_func="tanh", second_act_func="softplus", task="discrete", constrain_lambda=True)

# use accuracy as the metric to monitor during training
def compute_acc(model, val_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model.logits(xb)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return correct / max(total, 1)

# Define training configurations
optim_cfg = OptimConfig(
    optim="adam", 
    lr=1e-3, 
    weight_decay=1e-4
)

train_cfg = TrainConfig(
    max_epochs=200,
    batch_size=32,
    early_stop=True,
    patience=15,
    device=device,
    log_every=10,
    mode="max",
    seed=seed,
    verbose=True,
    monitor_name="val_acc",
)


export_cfg = ExportConfig(
    enabled=True,
    df = feature_df,
    model_path="breast_cancer_model_structure.txt",
)


# train the model using DiscreteTrainer, which is designed for classification tasks.
trainer = DiscreteTrainer(
    model=model,
    optim_cfg=optim_cfg,
    train_cfg=train_cfg,
    export_cfg=export_cfg,
    monitor_fn=compute_acc,
)

result = trainer.fit(x_train, y_train, x_val, y_val)

[Epoch 0010] train_loss=0.577027 val_loss=0.545751
[Epoch 0020] train_loss=0.178911 val_loss=0.144460
[Epoch 0030] train_loss=0.091886 val_loss=0.088370
Early stopping at epoch 30, best_epoch=15, best_val_acc=0.985401


### inference

In [7]:
from blnetwork.inference import predict_class_discrete, predict_proba_discrete

y_pred = predict_class_discrete(trainer.model, x_test).cpu().numpy().flatten()
y_true = y_test.cpu().numpy().flatten()
y_proba = predict_proba_discrete(trainer.model, x_test).cpu().numpy()[:, 1]

# compute and print evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc_score = roc_auc_score(y_true, y_proba)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"ROC AUC: {roc_auc_score:.4f}")

Accuracy: 0.9562
Precision: 0.8966
Recall: 1.0000
F1-score: 0.9455
ROC AUC: 0.9846
